In [ ]:
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
warnings.filterwarnings("ignore")
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

In [ ]:
DATA_DIR = Path(r"D:\Datasets\Geolife Trajectories 1.3\Data")  # EDIT THIS PATH

def load_plt_file(path):
    df=pd.read_csv(path, skiprows=6, header=None,
        names=["latitude","longitude","altitude","date_days","date","time"],
        usecols=[0,1,2,4,5])
    df["timestamp"]=pd.to_datetime(df["date"].astype(str)+" "+df["time"].astype(str), errors="coerce")
    for col in ["latitude","longitude","altitude"]: df[col]=pd.to_numeric(df[col], errors="coerce")
    df=df.dropna(subset=["latitude","longitude","altitude","timestamp"])
    df=df[df.latitude.between(-90,90)&df.longitude.between(-180,180)&(df.altitude>-1000)]
    return df.sort_values("timestamp").reset_index(drop=True)

def load_user(folder):
    parts=[load_plt_file(p) for p in folder.rglob("*.plt")]
    parts=[d for d in parts if len(d)]
    return pd.concat(parts,ignore_index=True).sort_values("timestamp").reset_index(drop=True) if parts else pd.DataFrame()

if not DATA_DIR.exists(): raise FileNotFoundError(f"Set DATA_DIR correctly: {DATA_DIR}")
trajectories={}
for folder in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    d=load_user(folder)
    if len(d)>=30: trajectories[folder.name]=d
    if len(trajectories)==3: break
if len(trajectories)<3: raise ValueError("Need at least 3 users with 30+ valid observations.")
print({u:len(d) for u,d in trajectories.items()})
splits={}
for uid,d in trajectories.items():
    cut=int(.8*len(d)); splits[uid]={"train":d.iloc[:cut].copy(),"test":d.iloc[cut:].copy()}
    print(uid, "train",cut,"test",len(d)-cut)

In [ ]:
def cluster_trajectories(split_dict, k=10):
    train_coords=np.vstack([v["train"][["latitude","longitude"]].to_numpy() for v in split_dict.values()])
    km=KMeans(n_clusters=k,random_state=RANDOM_STATE,n_init=10).fit(train_coords)
    data={}
    for uid,parts in split_dict.items():
        data[uid]={}
        for name in ["train","test"]:
            df=parts[name].copy(); df["cluster"]=km.predict(df[["latitude","longitude"]])
            data[uid][name]=df
    return km,data

kmeans, clustered=cluster_trajectories(splits,10)
centroids=kmeans.cluster_centers_
print(pd.DataFrame(centroids,columns=["latitude","longitude"]))
plt.figure(figsize=(9,6))
for uid,parts in clustered.items():
    d=parts["train"]; plt.scatter(d.longitude,d.latitude,c=d.cluster,cmap="tab20",s=8,alpha=.45)
plt.scatter(centroids[:,1],centroids[:,0],marker="X",s=150,c="black",label="Centroids")
plt.xlabel("Longitude");plt.ylabel("Latitude");plt.title("GeoLife K-Means geographical clusters");plt.legend();plt.grid(alpha=.2);plt.show()

In [ ]:
class CategoricalHMM:
    def __init__(self,n_states,n_symbols,n_iter=25,smoothing=1e-6,seed=42):
        self.n_states=n_states; self.n_symbols=n_symbols; self.n_iter=n_iter; self.smoothing=smoothing; self.seed=seed
    def norm(self,a,axis=None):
        s=np.sum(a,axis=axis,keepdims=True); return a/np.maximum(s,1e-300)
    def init(self):
        rng=np.random.default_rng(self.seed)
        self.pi=self.norm(rng.random(self.n_states)+self.smoothing)
        self.A=self.norm(rng.random((self.n_states,self.n_states))+self.smoothing,axis=1)
        self.B=self.norm(rng.random((self.n_states,self.n_symbols))+self.smoothing,axis=1)
    def forward_scaled(self,obs):
        T=len(obs); alpha=np.zeros((T,self.n_states)); scales=np.zeros(T)
        alpha[0]=self.pi*self.B[:,obs[0]]; scales[0]=max(alpha[0].sum(),1e-300); alpha[0]/=scales[0]
        for t in range(1,T):
            alpha[t]=(alpha[t-1]@self.A)*self.B[:,obs[t]]
            scales[t]=max(alpha[t].sum(),1e-300); alpha[t]/=scales[t]
        return alpha,scales
    def fit(self,sequences):
        sequences=[np.asarray(s,dtype=int) for s in sequences if len(s)]
        if not sequences: raise ValueError("No training sequences")
        self.init()
        for _ in range(self.n_iter):
            pi_sum=np.zeros(self.n_states); an=np.zeros_like(self.A); ad=np.zeros(self.n_states)
            bn=np.zeros_like(self.B); bd=np.zeros(self.n_states)
            for obs in sequences:
                alpha,sc=self.forward_scaled(obs); T=len(obs); beta=np.ones_like(alpha)
                for t in range(T-2,-1,-1): beta[t]=(self.A@(self.B[:,obs[t+1]]*beta[t+1]))/max(sc[t+1],1e-300)
                gamma=self.norm(alpha*beta,axis=1); pi_sum+=gamma[0]
                for t in range(T-1):
                    xi=alpha[t][:,None]*self.A*(self.B[:,obs[t+1]]*beta[t+1])[None,:]
                    xi/=max(xi.sum(),1e-300); an+=xi; ad+=gamma[t]
                for t,symbol in enumerate(obs): bn[:,symbol]+=gamma[t]; bd+=gamma[t]
            self.pi=self.norm(pi_sum+self.smoothing)
            self.A=self.norm(an+self.smoothing,axis=1); self.B=self.norm(bn+self.smoothing,axis=1)
        return self
    def predict_next_probs(self,obs):
        alpha,_=self.forward_scaled(np.asarray(obs,dtype=int))
        state_now=alpha[-1]; state_next=state_now@self.A; symbol_next=state_next@self.B
        return state_now,state_next,symbol_next/max(symbol_next.sum(),1e-300)
    def viterbi(self,obs):
        obs=np.asarray(obs,dtype=int); T=len(obs)
        lp=np.log(self.pi+1e-300); la=np.log(self.A+1e-300); lb=np.log(self.B+1e-300)
        delta=np.zeros((T,self.n_states)); psi=np.zeros((T,self.n_states),dtype=int)
        delta[0]=lp+lb[:,obs[0]]
        for t in range(1,T):
            scores=delta[t-1][:,None]+la; psi[t]=np.argmax(scores,axis=0)
            delta[t]=np.max(scores,axis=0)+lb[:,obs[t]]
        path=np.zeros(T,dtype=int); path[-1]=np.argmax(delta[-1])
        for t in range(T-2,-1,-1): path[t]=psi[t+1,path[t+1]]
        return path

In [ ]:
def haversine_m(lat1,lon1,lat2,lon2):
    R=6371000.; lat1,lon1,lat2,lon2=np.radians([lat1,lon1,lat2,lon2])
    a=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))
def evaluate(model,test_obs,test_coords,centers):
    probs_list=[];pred=[];true=[];dist=[]; start=max(1,len(test_obs)-100)
    for i in range(start,len(test_obs)):
        _,_,p=model.predict_next_probs(test_obs[:i]); y=int(test_obs[i]); yp=int(np.argmax(p))
        probs_list.append(p);true.append(y);pred.append(yp)
        dist.append(haversine_m(*test_coords[i],*centers[yp]))
    if not true: return dict(top1=np.nan,top3=np.nan,precision=np.nan,recall=np.nan,f1=np.nan,haversine_m=np.nan,preds=[],truth=[],probs=[])
    top3=np.mean([y in np.argsort(p)[-min(3,len(p)): ] for y,p in zip(true,probs_list)])
    labels=np.arange(len(centers))
    return dict(top1=accuracy_score(true,pred),top3=top3,
        precision=precision_score(true,pred,labels=labels,average="macro",zero_division=0),
        recall=recall_score(true,pred,labels=labels,average="macro",zero_division=0),
        f1=f1_score(true,pred,labels=labels,average="macro",zero_division=0),
        haversine_m=float(np.mean(dist)),preds=pred,truth=true,probs=probs_list)

models={}; results={}
for uid,parts in clustered.items():
    model=CategoricalHMM(5,10,n_iter=25).fit([parts["train"].cluster.to_numpy()])
    models[uid]=model; results[uid]=evaluate(model,parts["test"].cluster.to_numpy(),parts["test"][["latitude","longitude"]].to_numpy(),centroids)
display(pd.DataFrame(results).T[["top1","top3","precision","recall","f1","haversine_m"]])

uid=list(models)[0]; obs=clustered[uid]["train"].cluster.to_numpy(); path=models[uid].viterbi(obs)
print("Viterbi hidden states first 25:",path[:25])
_,_,next_probs=models[uid].predict_next_probs(obs); predicted=int(np.argmax(next_probs))
print("Predicted next cluster:",predicted,"centroid lat/lon:",centroids[predicted],"distribution:",np.round(next_probs,3))

In [ ]:
model=models[uid]
plt.figure(figsize=(7,6));sns.heatmap(model.A,annot=True,fmt=".2f",cmap="Blues",xticklabels=[f"S{i}" for i in range(model.n_states)],yticklabels=[f"S{i}" for i in range(model.n_states)])
plt.xlabel("Next hidden state");plt.ylabel("Current hidden state");plt.title("HMM hidden-state transition probabilities");plt.show()

In [ ]:
r=results[uid]
if r["truth"]:
    actual=centroids[np.array(r["truth"])]; predicted_xy=centroids[np.array(r["preds"])]
    plt.figure(figsize=(8,6));plt.scatter(actual[:,1],actual[:,0],label="Actual centroids",alpha=.7,s=30)
    plt.scatter(predicted_xy[:,1],predicted_xy[:,0],marker="x",label="Predicted centroids",alpha=.7,s=30)
    plt.xlabel("Longitude");plt.ylabel("Latitude");plt.title("Actual vs predicted next locations");plt.legend();plt.grid(alpha=.2);plt.show()

In [ ]:
rows=[]
for k in [5,10,15,20]:
    km,data=cluster_trajectories(splits,k)
    for user,parts in data.items():
        hm=CategoricalHMM(5,k,n_iter=20).fit([parts["train"].cluster.to_numpy()])
        t=time.perf_counter(); met=evaluate(hm,parts["test"].cluster.to_numpy(),parts["test"][["latitude","longitude"]].to_numpy(),km.cluster_centers_); elapsed=time.perf_counter()-t
        rows.append({"K clusters":k,"User":user,"Top-1":met["top1"],"Top-3":met["top3"],"Haversine m":met["haversine_m"],"Execution s":elapsed})
exp_k=pd.DataFrame(rows); sum_k=exp_k.groupby("K clusters",as_index=False).mean(numeric_only=True);display(sum_k)
plt.figure(figsize=(7,4));plt.plot(sum_k["K clusters"],sum_k["Top-1"],marker="o",label="Top-1");plt.plot(sum_k["K clusters"],sum_k["Top-3"],marker="s",label="Top-3")
plt.xlabel("K clusters");plt.ylabel("Accuracy");plt.title("Experiment 1: Cluster count impact");plt.legend();plt.grid(alpha=.3);plt.show()

In [ ]:
rows=[]
for fraction in [.50,.60,.80]:
    ss={}
    for user,d in trajectories.items():
        cut=int(len(d)*fraction);ss[user]={"train":d.iloc[:cut].copy(),"test":d.iloc[cut:].copy()}
    km,data=cluster_trajectories(ss,10)
    for user,parts in data.items():
        if len(parts["train"])<10 or len(parts["test"])<2:continue
        hm=CategoricalHMM(5,10,n_iter=20).fit([parts["train"].cluster.to_numpy()])
        met=evaluate(hm,parts["test"].cluster.to_numpy(),parts["test"][["latitude","longitude"]].to_numpy(),km.cluster_centers_)
        rows.append({"Training fraction":fraction,"User":user,"Top-1":met["top1"],"Top-3":met["top3"],"Haversine m":met["haversine_m"]})
exp_len=pd.DataFrame(rows);sum_len=exp_len.groupby("Training fraction",as_index=False).mean(numeric_only=True);display(sum_len)
plt.figure(figsize=(7,4));plt.plot(sum_len["Training fraction"]*100,sum_len["Top-1"],marker="o");plt.xlabel("Training trajectory (%)");plt.ylabel("Top-1 accuracy");plt.title("Experiment 2: Training length impact");plt.grid(alpha=.3);plt.show()

In [ ]:
rows=[]
for n_states in [2,3,5,8]:
    for user,parts in clustered.items():
        hm=CategoricalHMM(n_states,10,n_iter=20).fit([parts["train"].cluster.to_numpy()])
        met=evaluate(hm,parts["test"].cluster.to_numpy(),parts["test"][["latitude","longitude"]].to_numpy(),centroids)
        rows.append({"Hidden states":n_states,"User":user,"Top-1":met["top1"],"Top-3":met["top3"],"Haversine m":met["haversine_m"]})
exp_states=pd.DataFrame(rows);sum_states=exp_states.groupby("Hidden states",as_index=False).mean(numeric_only=True);display(sum_states)
plt.figure(figsize=(7,4));plt.plot(sum_states["Hidden states"],sum_states["Top-1"],marker="o",label="Top-1");plt.plot(sum_states["Hidden states"],sum_states["Top-3"],marker="s",label="Top-3");plt.xlabel("Hidden states");plt.ylabel("Accuracy");plt.title("Experiment 3: Hidden-state impact");plt.legend();plt.grid(alpha=.3);plt.show()

In [ ]:
# User-specific models were fitted above. Shared HMM uses each user's training sequence as a distinct sequence.
shared=CategoricalHMM(5,10,n_iter=25).fit([p["train"].cluster.to_numpy() for p in clustered.values()])
rows=[]
for user,parts in clustered.items():
    individual=results[user]
    shared_met=evaluate(shared,parts["test"].cluster.to_numpy(),parts["test"][["latitude","longitude"]].to_numpy(),centroids)
    rows.extend([{"User":user,"Model":"User-specific","Top-1":individual["top1"],"Top-3":individual["top3"],"Haversine m":individual["haversine_m"]},
                 {"User":user,"Model":"Shared","Top-1":shared_met["top1"],"Top-3":shared_met["top3"],"Haversine m":shared_met["haversine_m"]}])
exp_shared=pd.DataFrame(rows);display(exp_shared)
display(exp_shared.groupby("Model",as_index=False).mean(numeric_only=True))

In [ ]:
def horizon_probs(model,history,h):
    alpha,_=model.forward_scaled(np.asarray(history,dtype=int)); state=alpha[-1].copy()
    for _ in range(h): state=state@model.A
    p=state@model.B;return p/max(p.sum(),1e-300)
rows=[]
for h in [1,3,5]:
    for user,parts in clustered.items():
        train=parts["train"].cluster.to_numpy(); test=parts["test"].cluster.to_numpy(); coords=parts["test"][["latitude","longitude"]].to_numpy(); hm=models[user]
        correct=[];dist=[]
        for i in range(1,len(test)-h+1):
            history=np.concatenate([train,test[:i]]);p=horizon_probs(hm,history,h);pred=int(np.argmax(p));actual=int(test[i+h-1])
            correct.append(pred==actual);dist.append(haversine_m(*coords[i+h-1],*centroids[pred]))
        rows.append({"Horizon steps":h,"User":user,"Top-1":np.mean(correct) if correct else np.nan,"Haversine m":np.mean(dist) if dist else np.nan})
exp_h=pd.DataFrame(rows);sum_h=exp_h.groupby("Horizon steps",as_index=False).mean(numeric_only=True);display(sum_h)
plt.figure(figsize=(7,4));plt.plot(sum_h["Horizon steps"],sum_h["Top-1"],marker="o");plt.xlabel("Prediction horizon (steps)");plt.ylabel("Top-1 accuracy");plt.title("Experiment 5: Prediction horizon impact");plt.grid(alpha=.3);plt.show()

In [ ]:
print("Experiment 1 — cluster count");display(sum_k)
print("Experiment 2 — training sequence length");display(sum_len)
print("Experiment 3 — hidden states");display(sum_states)
print("Experiment 4 — user-specific vs shared");display(exp_shared)
print("Experiment 5 — prediction horizon");display(sum_h)